<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/GRUPO_4_(Pol%C3%ADtica_Monet%C3%A1ria)_VOLUME_DE_CR%C3%89DITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
!pip install python-bcb -q

In [69]:
import pandas as pd
from bcb import sgs

In [70]:
# Séries do SGS/BCB para saldo das operações de crédito (em R$ milhões)
series_credito = {
    'Crédito Total': 20539,
    'Pessoa Jurídica (PJ) - Total': 20540,
    'PJ - Recursos Livres': 20543,
    'PJ - Recursos Direcionados': 20594,
    'Pessoa Física (PF) - Total': 20541,
    'PF - Recursos Livres': 20570,
    'PF - Recursos Direcionados': 20606,
    'PF - Crédito Rural': 20607,
    'PF - Financiamento Imobiliário': 20610
}

print(f"Total de séries mapeadas: {len(series_credito)}")

Total de séries mapeadas: 9


In [71]:
list(series_credito.keys())

['Crédito Total',
 'Pessoa Jurídica (PJ) - Total',
 'PJ - Recursos Livres',
 'PJ - Recursos Direcionados',
 'Pessoa Física (PF) - Total',
 'PF - Recursos Livres',
 'PF - Recursos Direcionados',
 'PF - Crédito Rural',
 'PF - Financiamento Imobiliário']

In [72]:
df_bruto = sgs.get(series_credito, start='2024-01-01')
df_bruto.head()

,Crédito Total,Pessoa Jurídica (PJ) - Total,PJ - Recursos Livres,PJ - Recursos Direcionados,Pessoa Física (PF) - Total,PF - Recursos Livres,PF - Recursos Direcionados,PF - Crédito Rural,PF - Financiamento Imobiliário
Date,,,,,,,,,
2024-01-01,5794707,2234362,1421476,812886,3560345,1947518,1612827,209272,102284
2024-02-01,5815954,2235302,1423312,811990,3580652,1953857,1626796,215763,103079
2024-03-01,5899742,2285775,1470733,815043,3613967,1969433,1644534,223049,104355
2024-04-01,5915538,2268879,1452275,816604,3646659,1989763,1656896,227186,105461
2024-05-01,5953960,2278309,1453046,825262,3675652,2008616,1667035,230154,107201


In [73]:
print("Linhas e colunas:", df_bruto.shape)
df_bruto.tail(3)

Linhas e colunas: (33, 9)


,Crédito Total,Pessoa Jurídica (PJ) - Total,PJ - Recursos Livres,PJ - Recursos Direcionados,Pessoa Física (PF) - Total,PF - Recursos Livres,PF - Recursos Direcionados,PF - Crédito Rural,PF - Financiamento Imobiliário
Date,,,,,,,,,
2026-07-01,7362252,2729062,1585565,1143497,4633191,2551651,2081540,240541,167762
2026-08-01,7412361,2731883,1577667,1154216,4680478,2577010,2103468,237696,169272
2026-09-01,7444208,2759442,1605274,1154169,4684766,2582163,2102603,226565,170988


In [74]:
df_bruto.isna().sum()

,0
Crédito Total,0
Pessoa Jurídica (PJ) - Total,0
PJ - Recursos Livres,0
PJ - Recursos Direcionados,0
Pessoa Física (PF) - Total,0
PF - Recursos Livres,0
PF - Recursos Direcionados,0
PF - Crédito Rural,0
PF - Financiamento Imobiliário,0


In [75]:
def criar_tabela_base(df):
    base = df.reset_index().melt(
        id_vars='Date',
        var_name='indicador',
        value_name='valor'
    )
    base['data'] = base['Date'].dt.strftime('%Y%m01')
    base = base.sort_values(['indicador', 'Date']).reset_index(drop=True)
    return base

In [76]:
tabela_base = criar_tabela_base(df_bruto)
tabela_base.head(10)

,Date,indicador,valor,data
0,2024-01-01,Crédito Total,5794707,20240101
1,2024-02-01,Crédito Total,5815954,20240201
2,2024-03-01,Crédito Total,5899742,20240301
3,2024-04-01,Crédito Total,5915538,20240401
4,2024-05-01,Crédito Total,5953960,20240501
5,2024-06-01,Crédito Total,6041898,20240601
6,2024-07-01,Crédito Total,6069205,20240701
7,2024-08-01,Crédito Total,6135298,20240801
8,2024-09-01,Crédito Total,6216528,20240901
9,2024-10-01,Crédito Total,6275117,20241001


In [77]:
tabela_base[['data', 'indicador', 'valor']].tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,2654220
289,20260201,Pessoa Jurídica (PJ) - Total,2656484
290,20260301,Pessoa Jurídica (PJ) - Total,2687535
291,20260401,Pessoa Jurídica (PJ) - Total,2685397
292,20260501,Pessoa Jurídica (PJ) - Total,2708154
293,20260601,Pessoa Jurídica (PJ) - Total,2750759
294,20260701,Pessoa Jurídica (PJ) - Total,2729062
295,20260801,Pessoa Jurídica (PJ) - Total,2731883
296,20260901,Pessoa Jurídica (PJ) - Total,2759442


In [78]:
# Nota: como o saldo de crédito é variável de estoque (posição em R$ milhões),
# não somamos os saldos ao longo dos meses (T1 de fluxo).
# Calculamos a variação % frente ao saldo de dezembro do ano anterior.

def calcular_var_ano(base):
    df = base.copy()
    df['ano'] = df['Date'].dt.year

    dezembro = df[df['Date'].dt.month == 12][['indicador', 'ano', 'valor']].copy()
    dezembro['ano'] = dezembro['ano'] + 1
    dezembro = dezembro.rename(columns={'valor': 'valor_dez_ant'})

    df = df.merge(dezembro, on=['indicador', 'ano'], how='left')
    df['valor'] = ((df['valor'] / df['valor_dez_ant']) - 1) * 100

    return df[['data', 'indicador', 'valor']].dropna()

In [79]:
tab_interm_1 = calcular_var_ano(tabela_base)
tab_interm_1.tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,-1.758685
289,20260201,Pessoa Jurídica (PJ) - Total,-1.674887
290,20260301,Pessoa Jurídica (PJ) - Total,-0.525588
291,20260401,Pessoa Jurídica (PJ) - Total,-0.604723
292,20260501,Pessoa Jurídica (PJ) - Total,0.237588
293,20260601,Pessoa Jurídica (PJ) - Total,1.814538
294,20260701,Pessoa Jurídica (PJ) - Total,1.011461
295,20260801,Pessoa Jurídica (PJ) - Total,1.115876
296,20260901,Pessoa Jurídica (PJ) - Total,2.135924


In [80]:
# Para variável de estoque, a variação em 12 meses compara o saldo do mês
# com o saldo do mesmo mês no ano anterior (12 meses antes).

def calcular_var_12m(base):
    df = base.copy()
    df['valor'] = df.groupby('indicador')['valor'].pct_change(periods=12) * 100
    return df[['data', 'indicador', 'valor']].dropna()

In [81]:
tab_interm_2 = calcular_var_12m(tabela_base)
tab_interm_2.tail(9)

,data,indicador,valor
288,20260101,Pessoa Jurídica (PJ) - Total,8.249044
289,20260201,Pessoa Jurídica (PJ) - Total,7.239486
290,20260301,Pessoa Jurídica (PJ) - Total,7.576187
291,20260401,Pessoa Jurídica (PJ) - Total,6.884621
292,20260501,Pessoa Jurídica (PJ) - Total,6.927210
293,20260601,Pessoa Jurídica (PJ) - Total,7.970584
294,20260701,Pessoa Jurídica (PJ) - Total,7.257585
295,20260801,Pessoa Jurídica (PJ) - Total,7.289815
296,20260901,Pessoa Jurídica (PJ) - Total,6.549691


In [82]:
ultimas_3_datas = sorted(tabela_base['data'].unique())[-3:]
print("Últimas 3 datas:", ultimas_3_datas)

Últimas 3 datas: ['20260701', '20260801', '20260901']


In [83]:
def montar_painel_wide(base, t_ano, t_12m, ordem_categorias):
    ultimas = sorted(base['data'].unique())[-3:]
    ultima_data = ultimas[-1]

    wide = base[base['data'].isin(ultimas)].pivot(
        index='indicador',
        columns='data',
        values='valor'
    )

    col_ano = t_ano[t_ano['data'] == ultima_data].set_index('indicador')[['valor']]
    col_ano = col_ano.rename(columns={'valor': f'Var. no ano ({ultima_data}) %'})

    col_12m = t_12m[t_12m['data'] == ultima_data].set_index('indicador')[['valor']]
    col_12m = col_12m.rename(columns={'valor': f'Var. 12m ({ultima_data}) %'})

    painel = wide.join([col_ano, col_12m])
    return painel.reindex(ordem_categorias).round(2)

In [84]:
ordem = list(series_credito.keys())
painel_final = montar_painel_wide(tabela_base, tab_interm_1, tab_interm_2, ordem)

In [85]:
display(painel_final)

,20260701,20260801,20260901,Var. no ano (20260901) %,Var. 12m (20260901) %
indicador,,,,,
Crédito Total,7362252,7412361,7444208,4.30,8.63
Pessoa Jurídica (PJ) - Total,2729062,2731883,2759442,2.14,6.55
PJ - Recursos Livres,1585565,1577667,1605274,-2.04,1.22
PJ - Recursos Direcionados,1143497,1154216,1154169,8.58,14.97
Pessoa Física (PF) - Total,4633191,4680478,4684766,5.62,9.90
PF - Recursos Livres,2551651,2577010,2582163,5.02,10.03
PF - Recursos Direcionados,2081540,2103468,2102603,6.37,9.73
PF - Crédito Rural,240541,237696,226565,-1.49,-1.21
PF - Financiamento Imobiliário,167762,169272,170988,9.70,16.98


In [86]:
data_ref = ultimas_3_datas[-1]
total_cred = painel_final.loc['Crédito Total', data_ref]
total_pf = painel_final.loc['Pessoa Física (PF) - Total', data_ref]
total_pj = painel_final.loc['Pessoa Jurídica (PJ) - Total', data_ref]

print(f"Data de referência: {data_ref}")
print(f"Participação PF: {(total_pf / total_cred) * 100:.2f}%")
print(f"Participação PJ: {(total_pj / total_cred) * 100:.2f}%")

Data de referência: 20260901
Participação PF: 62.93%
Participação PJ: 37.07%


In [87]:
!pip install Plotly